In [6]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import RobustScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

class OutlierDiagnoser:
    '''To learn normal relationships'''
    def __init__(self, features):
        self.features = features
        self.diagnostic_models = {}
        self.scalers = {} #know more about this class stuff

        print("Diagnosis Init")

    def fit(self, clean_df):
        '''Neural training to make model learn about what actually is normal'''
        for target_values in self.features:
            X = clean_df.drop(columns=[target_values])
            y = clean_df[target_values]

            scaler = RobustScaler()
            scaledx = scaler.fit_transform(X)

            self.scalers[target_values] = scaler

            #neural network model for training them
            model = Sequential([
                Input(shape=(scaledx.shape[1],)),
                Dense(16, activation = 'relu'),
                Dense(8, activation='relu'),
                Dense(1)
            ])

            #compiling the neural network model
            model.compile(optimizer = 'adam', loss = 'mean_squared_error')
            model.fit(scaledx, y, epochs=50, verbose=0)

            #saving the model for future use
            self.diagnostic_models[target_values] = model

            print("model is now ready")

    def dmlp(self, outlier_row):
        '''
        For a single outlier row, calculates the prediction error for each feature
        and returns the name of the feature with the largest error.
        '''
        errors = {} #for collecting the errors

        for target_feature in self.features:

            model = self.diagnostic_models[target_feature]
            scaler = self.scalers[target_feature]

            actual_value = outlier_row[target_feature] #getting the values from the outlier row

            #features for value prediction
            prediction_features = outlier_row.drop(target_feature).values.reshape(1, -1)
            scaled_prediction_features = scaler.transform(prediction_features)

            #predict the alternate features
            predicted_value = model.predict(scaled_prediction_features, verbose=0)[0][0]

            error = abs(predicted_value - actual_value)
            errors[target_feature] = error

        # Return the feature name with the highest error
        most_problematic_feature = max(errors, key=errors.get)
        return most_problematic_feature

def repair_outlier_feature(df_full, normal_df, outlier_row_index, column_to_fix):

    #reparing single featire for a specific outlier row

    features = [col for col in normal_df.columns if col != column_to_fix]

    X_train = normal_df[features]
    y_train = normal_df[column_to_fix]
    X_predict = df_full.loc[[outlier_row_index]][features]

    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_predict_scaled = scaler.transform(X_predict)

    model = Sequential([
        Input(shape=(len(features),)),
        Dense(32, activation = 'relu'),
        Dense(16, activation = 'relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    model.fit(X_train_scaled, y_train, epochs=50, verbose=0)

    predicted_value = model.predict(X_predict_scaled, verbose=0)[0][0]

    print(f"--> Prediction for replacement: {predicted_value:.2f}")
    return predicted_value

In [7]:
# Create a sample DataFrame with some outliers
data = {'feature1': [1, 2, 3, 4, 5, 100],
        'feature2': [10, 20, 30, 40, 50, 200],
        'feature3': [100, 200, 300, 400, 500, 30]}
df = pd.DataFrame(data)

# Assume the last row is an outlier for demonstration
outlier_row_index = 5
outlier_row = df.loc[outlier_row_index]
normal_df = df.drop(outlier_row_index)

# Initialize and fit the OutlierDiagnoser
features_to_diagnose = ['feature1', 'feature2', 'feature3']
diagnoser = OutlierDiagnoser(features_to_diagnose)
diagnoser.fit(normal_df)

# Diagnose the outlier row
most_problematic_feature = diagnoser.dmlp(outlier_row)
print(f"The most problematic feature in the outlier row is: {most_problematic_feature}")

# Repair the outlier feature
repaired_value = repair_outlier_feature(df, normal_df, outlier_row_index, most_problematic_feature)
print(f"The repaired value for '{most_problematic_feature}' is: {repaired_value}")

# Optionally, update the DataFrame with the repaired value
df.loc[outlier_row_index, most_problematic_feature] = repaired_value

print("\nDataFrame after potential repair:")
display(df)

Diagnosis Init
model is now ready
model is now ready
model is now ready


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RobustScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RobustScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RobustScaler was fitted with feature names
  warnings.warn(


The most problematic feature in the outlier row is: feature2
--> Prediction for replacement: 22.26
The repaired value for 'feature2' is: 22.256303787231445

DataFrame after potential repair:


/tmp/ipython-input-7-2629432510.py:26: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '22.256303787231445' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[outlier_row_index, most_problematic_feature] = repaired_value


,feature1,feature2,feature3
0,1,10.000000,100
1,2,20.000000,200
2,3,30.000000,300
3,4,40.000000,400
4,5,50.000000,500
5,100,22.256304,30
